# Literature methodology and reference generation

Supports the survey *Quantum Networks Still Need Watching* (Section 2, Survey Methodology). **No manuscript files are needed.**
The notebook (1) searches OpenAlex and arXiv, (2) de-duplicates, (3) screens titles and abstracts with transparent keyword rules,
(4) checks the recall of the search against the curated reference list stored in `data/curated_references.csv`,
(5) computes the corpus statistics quoted in the paper, (6) **generates `ref.bib`** from the curated list and verifies every entry
against Crossref and arXiv, and (7) writes the methodology paragraph with the numbers filled in.

All outputs are written to `outputs/`. Set `REPO_URL` below (or run the notebook from a clone of the repository).

In [1]:
# Cell 1: install dependencies
!pip -q install rapidfuzz tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 24.3 MB/s eta 0:00:00


In [2]:
# Cell 2: configuration
import os, re, json, time, zipfile, shutil, subprocess, unicodedata, collections
import xml.etree.ElementTree as ET
import requests, pandas as pd
from tqdm.auto import tqdm
from rapidfuzz import fuzz

REPO_URL = "https://github.com/<your-user>/<your-repo>.git"   # <- set this after creating the GitHub repository
EMAIL = "you@example.com"          # polite-pool e-mail for OpenAlex and Crossref (use your own)
OPENALEX_API_KEY = ""              # free key from https://openalex.org/settings/api (or a Colab secret named OPENALEX_API_KEY)
try:
    from google.colab import userdata
    OPENALEX_API_KEY = OPENALEX_API_KEY or userdata.get("OPENALEX_API_KEY")
except Exception:
    pass

DATE_FROM, DATE_TO = "2015-01-01", "2026-12-31"     # detection-oriented studies
MAX_PER_STRING = 3000                                # safety cap per search string and source
INCLUDE_S4 = False                                   # optional 4th string that improves recall (see Cell 8); off to reproduce the paper
VERIFY_ONLINE = True                                 # Cell 10: query Crossref/arXiv (False = build ref.bib offline)
MAX_AUTHORS = 6                                      # authors kept per bib entry before "and others"

SEARCH_STRINGS = {
 "S1_detection": '("quantum key distribution" OR "quantum network" OR "quantum communication") AND ("intrusion detection" OR "anomaly detection" OR "attack detection" OR "eavesdropping detection" OR QBER)',
 "S2_ml_qml":   '("quantum machine learning" OR "quantum neural network" OR "variational quantum") AND ("intrusion detection" OR "anomaly detection")',
 "S3_threats":  '("quantum network" OR "quantum repeater" OR QKD) AND (attack OR vulnerability OR "side channel" OR "Trojan horse" OR "man-in-the-middle")',
}
if INCLUDE_S4:
    SEARCH_STRINGS["S4_countermeasures"] = '(QKD OR "quantum key distribution") AND (countermeasure OR loophole OR hacking OR "Trojan horse" OR "machine learning")'

HEADERS = {"User-Agent": f"survey-methodology/1.0 (mailto:{EMAIL})"}
OUT = "outputs"; os.makedirs(OUT, exist_ok=True)
RUN_DATE = time.strftime("%Y-%m-%d")
print("run date:", RUN_DATE, "| OpenAlex key set:", bool(OPENALEX_API_KEY))

run date: 2026-10-18 | OpenAlex key set: True


In [3]:
# Cell 3: load the curated reference list from the repository (no upload needed)
CURATED_CANDIDATES = ["data/curated_references.csv", "../data/curated_references.csv", "repo/data/curated_references.csv"]
path = next((p for p in CURATED_CANDIDATES if os.path.exists(p)), None)
if path is None:
    if "<your-user>" in REPO_URL:
        raise RuntimeError("Set REPO_URL in Cell 2 to your GitHub repository, or run this notebook from a clone of it.")
    if os.path.exists("repo"): shutil.rmtree("repo")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
    path = "repo/data/curated_references.csv"
curated = pd.read_csv(path).fillna("")
curated["year"] = pd.to_numeric(curated.year, errors="coerce")
print(len(curated), "curated references loaded from", path)
display(curated[["key", "year", "role", "used_in", "title"]].head(5))

126 curated references loaded from repo/data/curated_references.csv


,key,year,role,used_in,title
0,Ref1,2018,core,intro;evaluation,Quantum internet: A vision for the road ahead
1,Ref2,2008,background,intro,The quantum internet
2,Ref3,2020,background,intro,Quantum internet: Networking challenges in dis...
3,Ref4,2026,background,intro,Quantum protocol architectures and secure cont...
4,Ref5,2026,core,intro;challenges,Device‐independent quantum key distribution: P...


In [4]:
# Cell 4: OpenAlex search
def inv_to_text(inv):
    if not inv: return ""
    pairs = [(i, w) for w, idxs in inv.items() for i in idxs]
    return " ".join(w for _, w in sorted(pairs))

def openalex_search(name, q):
    rows, total, page = [], None, 1
    sel = "id,doi,title,publication_year,type,language,primary_location,abstract_inverted_index,authorships,cited_by_count"
    while len(rows) < MAX_PER_STRING and page * 200 <= 10000:            # OpenAlex page limit: 10,000 results
        params = {"filter": f"title_and_abstract.search:{q},from_publication_date:{DATE_FROM},to_publication_date:{DATE_TO}",
                  "per-page": 200, "page": page, "select": sel, "mailto": EMAIL}
        if OPENALEX_API_KEY: params["api_key"] = OPENALEX_API_KEY
        r = requests.get("https://api.openalex.org/works", params=params, headers=HEADERS, timeout=60)
        if r.status_code != 200:
            print(name, "HTTP", r.status_code, r.text[:300])
            if r.status_code in (401, 403) and not OPENALEX_API_KEY:
                print("  -> create a free key at https://openalex.org/settings/api and set OPENALEX_API_KEY in Cell 2")
            break
        j = r.json()
        if total is None: total = j["meta"]["count"]
        results = j.get("results", [])
        if not results: break
        for w in results:
            loc = w.get("primary_location") or {}; auth = w.get("authorships") or []
            rows.append(dict(source="openalex", query=name, id=w["id"],
                doi=(w.get("doi") or "").replace("https://doi.org/", "").lower(),
                title=w.get("title") or "", year=w.get("publication_year"), type=w.get("type"), language=w.get("language"),
                venue=(loc.get("source") or {}).get("display_name"), abstract=inv_to_text(w.get("abstract_inverted_index")),
                first_author=(auth[0]["author"]["display_name"] if auth else ""), cited_by=w.get("cited_by_count")))
        if len(results) < 200: break
        page += 1; time.sleep(0.2)
    return rows[:MAX_PER_STRING], total

oa_rows, oa_total = [], {}
for name, q in SEARCH_STRINGS.items():
    rows, total = openalex_search(name, q)
    oa_rows += rows; oa_total[name] = total
    flag = "" if total is None or len(rows) >= total else "  <-- INCOMPLETE (check the API key)"
    print(f"OpenAlex {name}: retrieved {len(rows)} of {total} hits{flag}")

OpenAlex S1_detection: retrieved 955 of 955 hits
OpenAlex S2_ml_qml: retrieved 437 of 437 hits
OpenAlex S3_threats: retrieved 2792 of 2792 hits


In [5]:
# Cell 5: arXiv search
NS = {"a": "http://www.w3.org/2005/Atom", "x": "http://arxiv.org/schemas/atom"}
OSNS = "{http://a9.com/-/spec/opensearch/1.1/}"

def to_arxiv_query(q):
    toks = re.findall(r'"[^"]+"|\(|\)|\bAND\b|\bOR\b|\bNOT\b|[^\s()]+', q)
    out = [t if t in ("(", ")", "AND", "OR", "NOT") else "all:" + t for t in toks]
    return " ".join(out).replace("( ", "(").replace(" )", ")")

def arxiv_search(name, q):
    rows, start, page, total = [], 0, 100, None
    sq = f"({to_arxiv_query(q)}) AND submittedDate:[{DATE_FROM.replace('-','')}0000 TO {DATE_TO.replace('-','')}2359]"
    while start < MAX_PER_STRING:
        r = requests.get("http://export.arxiv.org/api/query", params=dict(search_query=sq, start=start, max_results=page, sortBy="relevance"),
                         headers=HEADERS, timeout=60)
        if r.status_code != 200:
            print(name, "HTTP", r.status_code); break
        root = ET.fromstring(r.text)
        if total is None: total = int(root.find(OSNS + "totalResults").text)
        entries = root.findall("a:entry", NS)
        if not entries: break
        for e in entries:
            doi = e.find("x:doi", NS); names = [a.find("a:name", NS).text for a in e.findall("a:author", NS)]
            rows.append(dict(source="arxiv", query=name, id=e.find("a:id", NS).text, doi=(doi.text.lower() if doi is not None else ""),
                title=re.sub(r"\s+", " ", e.find("a:title", NS).text or "").strip(), year=int(e.find("a:published", NS).text[:4]),
                type="preprint", language="en", venue="arXiv",
                abstract=re.sub(r"\s+", " ", e.find("a:summary", NS).text or "").strip(),
                first_author=names[0] if names else "", cited_by=None))
        start += page
        if start >= total: break
        time.sleep(3.1)                                                     # arXiv API etiquette
    return rows[:MAX_PER_STRING], total

ax_rows, ax_total = [], {}
for name, q in SEARCH_STRINGS.items():
    rows, total = arxiv_search(name, q)
    ax_rows += rows; ax_total[name] = total
    print(f"arXiv {name}: retrieved {len(rows)} of {total} hits")

arXiv S1_detection: retrieved 156 of 156 hits
arXiv S2_ml_qml: retrieved 73 of 73 hits
arXiv S3_threats: retrieved 647 of 647 hits


In [6]:
# Cell 6: merge and de-duplicate
df = pd.DataFrame(oa_rows + ax_rows)
n_identified = len(df)                                   # all records returned, including overlaps
norm_title = lambda t: re.sub(r"[^a-z0-9]", "", (t or "").lower())
df["ntitle"] = df.title.map(norm_title); df = df[df.ntitle.ne("")]
df["has_doi"] = df.doi.fillna("").ne(""); df["has_abs"] = df.abstract.fillna("").str.len() > 50
df["rank"] = (~df.has_doi).astype(int)*2 + (~df.has_abs).astype(int) + (df.source == "arxiv")*0.5
queries = df.groupby("ntitle")["query"].agg(lambda s: ";".join(sorted(set(s))))
sources = df.groupby("ntitle")["source"].agg(lambda s: ";".join(sorted(set(s))))
d1 = df.sort_values("rank").drop_duplicates("ntitle")                                   # same title
d1 = pd.concat([d1[d1.doi.fillna("") == ""], d1[d1.doi.fillna("") != ""].drop_duplicates("doi")])   # same DOI
n_after_exact = len(d1)

def fuzzy_dedup(d):                                      # arXiv preprint vs published version with a slightly different title
    d = d.copy(); d["year"] = pd.to_numeric(d.year, errors="coerce")
    by_year = collections.defaultdict(list)
    for i, r in d[d.source == "openalex"].iterrows(): by_year[r.year].append(r.ntitle)
    drop = set()
    for i, r in d[d.source == "arxiv"].iterrows():
        if pd.isna(r.year): continue
        if any(fuzz.ratio(r.ntitle, nt) >= 93 for y in (r.year-1, r.year, r.year+1) for nt in by_year.get(y, [])): drop.add(i)
    return d.drop(index=list(drop))

dd = fuzzy_dedup(d1)
dd["queries"] = dd.ntitle.map(queries); dd["sources"] = dd.ntitle.map(sources)
n_dedup = len(dd)
dd.drop(columns=["rank"]).to_csv(f"{OUT}/records_deduplicated.csv", index=False)
print(f"records identified: {n_identified} | after exact de-duplication: {n_after_exact} | after preprint de-duplication: {n_dedup}")

records identified: 5060 | after exact de-duplication: 3765 | after preprint de-duplication: 3764


In [7]:
# Cell 7: title and abstract screening in three tiers (rule-based, transparent)
BAD_TYPES = {"editorial","erratum","paratext","letter","retraction","peer-review","grant","dataset","supplementary-materials","libguides","reference-entry"}
Q = re.compile(r"quantum key distribution|\bqkd\b|quantum network|quantum communication|quantum internet|quantum repeater|entanglement distribution|\bbb84\b|\be91\b|\bbbm92\b|continuous[- ]variable quantum|cv-qkd|quantum link|quantum cryptograph", re.I)
DET = re.compile(r"\bdetect\w*|monitor\w*|anomal\w*|intrusion|countermeasure|machine learning|neural network|deep learning|classif\w*", re.I)
ATT = re.compile(r"attack\w*|eavesdrop\w*|hack\w*|vulnerab\w*|trojan|side[- ]channel|man[- ]in[- ]the[- ]middle|intrusion|tamper\w*|jamming|spoof\w*|intercept\w*|blinding|denial[- ]of[- ]service|\bddos\b", re.I)
PROOF = re.compile(r"security proof|finite[- ]key|composable|asymptotic|entropic|secret[- ]key rate|key rate|lower bound|upper bound|capacity|coherent attack|collective attack|general attack|\bproof\b", re.I)
DET_TITLE = re.compile(r"detect|monitor|machine learning|neural|deep learning|countermeasure", re.I)
QML = re.compile(r"quantum machine learning|\bqml\b|quantum neural|variational quantum|quantum kernel|quantum support vector|\bqsvm\b|hybrid quantum[- ]classical|quantum[- ]inspired|quantum autoencoder|quantum federated|quantum convolutional", re.I)
CYB = re.compile(r"intrusion detection|network intrusion|malware|\bddos\b|botnet|network traffic|nsl-kdd|cicids|unsw|attack detection|cyber[- ]?attack|cybersecurity|zero[- ]day|\bids\b", re.I)
ANOM = re.compile(r"anomaly detection", re.I)
NETCTX = re.compile(r"network|traffic|\biot\b|cyber|vehicular|smart grid|industrial control|\bscada\b", re.I)

def tier(r):
    t, ti = f"{r.title} {r.abstract}", r.title
    if r.type in BAD_TYPES: return "exclude", "document type"
    if isinstance(r.language, str) and r.language != "en": return "exclude", "language"
    if QML.search(t) and (CYB.search(t) or (ANOM.search(t) and NETCTX.search(t))): return "T2", "QML for intrusion/anomaly detection in a cyber or network context"
    if Q.search(t) and DET.search(t) and ATT.search(t) and (DET.search(ti) or ATT.search(ti)):
        if PROOF.search(ti) and not DET_TITLE.search(ti): return "exclude", "security proof or key-rate analysis"
        return "T1", "quantum communication + detection/ML + attack"
    if Q.search(t) and ATT.search(ti) and not PROOF.search(ti): return "T3", "quantum communication attack or vulnerability"
    return "exclude", "off-topic"

s = dd.copy(); s["abstract"] = s.abstract.fillna(""); s["title"] = s.title.fillna("")
res = s.apply(lambda r: pd.Series(tier(r), index=["tier", "reason"]), axis=1)
scr = pd.concat([s, res], axis=1); scr["cited_by"] = pd.to_numeric(scr.cited_by, errors="coerce")
scr.drop(columns=["rank"], errors="ignore").sort_values(["tier", "cited_by"], ascending=[True, False]).to_csv(f"{OUT}/screening_tiered.csv", index=False)
tcount = scr.tier.value_counts().reindex(["T1", "T2", "T3", "exclude"]).fillna(0).astype(int)
n_candidates = int(tcount[["T1", "T2", "T3"]].sum())
print(tcount.to_string()); print("candidates (T1+T2+T3):", n_candidates, "of", len(scr))
print("records without abstract:", int((scr.abstract.str.len() < 50).sum()), "| share of 2025-2026:", round(100*scr.year.isin([2025, 2026]).mean(), 1), "%")

tier
T1          414
T2          325
T3          155
exclude    2870
candidates (T1+T2+T3): 894 of 3764
records without abstract: 494 | share of 2025-2026: 52.2 %


In [8]:
# Cell 8: recall check against the curated list, and coverage gaps
norm = lambda t: re.sub(r"[^a-z0-9]", "", str(t).lower())
scr["nt"] = scr.title.map(norm); curated["nt"] = curated.title.map(norm)
in_corpus = set(scr.nt)

known = curated[curated.recall_set.astype(str).str.lower().eq("true")].copy()
known["found_in_search"] = known.nt.isin(in_corpus)
known["tier"] = known.nt.map(dict(zip(scr.nt, scr.tier)))
n_known, n_found = len(known), int(known.found_in_search.sum())
print(f"search recall on known relevant references: {n_found} of {n_known} ({100*n_found/n_known:.0f}%)")
print("not retrieved:", list(known[~known.found_in_search].key))
known[["key", "title", "found_in_search", "tier"]].to_csv(f"{OUT}/recall_check.csv", index=False)

cand = scr[scr.tier.isin(["T1", "T2"]) & ~scr.nt.isin(set(curated.nt))].copy()
cand["cited_by"] = cand.cited_by.fillna(0); cand["priority"] = cand.cited_by / (1 + (2026 - pd.to_numeric(cand.year, errors="coerce")))
cand["is_review"] = cand.title.str.contains("survey|review|overview|taxonomy", case=False)
gap = cand.sort_values("priority", ascending=False)
gap[["tier", "year", "cited_by", "priority", "is_review", "title", "venue", "doi", "first_author"]].head(80).to_csv(f"{OUT}/coverage_gap_candidates.csv", index=False)
display(gap[["tier", "year", "cited_by", "title"]].head(15))

search recall on known relevant references: 24 of 31 (77%)
not retrieved: ['Ref60', 'Ref61', 'Ref62', 'Ref63', 'Gupta2026', 'Long2023', 'Mafu2024']


,tier,year,cited_by,title
1258,T2,2024,115.0,Advancing cybersecurity and privacy with artif...
1090,T2,2022,105.0,Anomaly detection in high-energy physics using...
10,T1,2025,41.0,A Novel Approach Based on Quantum Key Distribu...
1106,T2,2024,49.0,Optimizing intrusion detection using intellige...
2366,T1,2023,64.0,Hacking measurement-device-independent quantum...
893,T2,2023,63.0,Benchmarking adversarially robust quantum mach...
1842,T1,2018,116.0,Homodyne-detector-blinding attack in continuou...
1950,T1,2016,140.0,Practical quantum private query with better pe...
1069,T2,2024,38.0,Quantum intrusion detection system using outli...
1056,T2,2024,38.0,Quantum Machine Learning for Advanced Threat D...


In [9]:
# Cell 9: corpus statistics and the methodology paragraph
y = curated.year
stats = dict(
    n_all=len(curated), n_core=int((curated.role == "core").sum()), n_other=int((curated.role != "core").sum()),
    n_new=int((y >= 2023).sum()), n_mid=int(y.between(2020, 2022).sum()), n_old=int((y <= 2019).sum()),
    n_old_bg=int(((y <= 2019) & (curated.role != "core")).sum()))
print(stats)

fmt = lambda n: f"{n:,}"
paragraph = (
 f"Literature was retrieved in {time.strftime('%B %Y')} through the OpenAlex and arXiv programming interfaces using the search strings in "
 f"Table~\\ref{{tab:search}}, restricted to {DATE_FROM[:4]}--{DATE_TO[:4]} for detection-oriented studies, and was complemented by backward and forward citation tracing; "
 f"foundational physics and protocol papers of any date were retained. The searches returned {fmt(n_identified)} records, and {fmt(n_dedup)} remained after de-duplication by DOI and title. "
 f"Keyword rules applied to titles and abstracts retained {n_candidates} candidate records: {tcount['T1']} on detection in quantum communication systems, "
 f"{tcount['T2']} on quantum machine learning for intrusion or anomaly detection, and {tcount['T3']} on attacks and vulnerabilities of quantum communication. "
 f"Candidates were prioritized by relevance, citation impact, and recency rather than read exhaustively. A check against {n_known} references already known to be relevant showed that the searches retrieved "
 f"{n_found} of them ({100*n_found/n_known:.0f}\\%), so recall is incomplete, and the other {n_known-n_found} were added manually. "
 f"The resulting corpus contains {stats['n_all']} references. Of these, {stats['n_core']} are cited in the threat, detection, evaluation, and challenge sections "
 f"(Sections~\\ref{{sec:threat_landscape}}--\\ref{{sec:challenges_future}}) and {stats['n_other']} are cited only in the introduction, this section, and the background. "
 f"By publication year, {stats['n_new']} references date from 2023--2026, {stats['n_mid']} from 2020--2022, and {stats['n_old']} from 2019 or earlier, "
 f"of which {stats['n_old_bg']} are cited only in the introduction and background.")
open(f"{OUT}/methodology_paragraph.tex", "w", encoding="utf-8").write(paragraph + "\n")
pd.DataFrame([("records identified", n_identified), ("after de-duplication", n_dedup), ("candidates after keyword screening", n_candidates),
              ("T1 detection in quantum communication", int(tcount["T1"])), ("T2 QML for intrusion/anomaly detection", int(tcount["T2"])),
              ("T3 attacks and vulnerabilities", int(tcount["T3"])), ("known relevant references", n_known), ("retrieved by the search", n_found),
              ("curated references cited in the paper", stats["n_all"])], columns=["stage", "n"]).to_csv(f"{OUT}/funnel_counts.csv", index=False)
print(paragraph)

{'n_all': 126, 'n_core': 74, 'n_other': 52, 'n_new': 51, 'n_mid': 19, 'n_old': 56, 'n_old_bg': 29}
Literature was retrieved in October 2026 through the OpenAlex and arXiv programming interfaces using the search strings in Table~\ref{tab:search}, restricted to 2015--2026 for detection-oriented studies, and was complemented by backward and forward citation tracing; foundational physics and protocol papers of any date were retained. The searches returned 5,060 records, and 3,764 remained after de-duplication by DOI and title. Keyword rules applied to titles and abstracts retained 894 candidate records: 414 on detection in quantum communication systems, 325 on quantum machine learning for intrusion or anomaly detection, and 155 on attacks and vulnerabilities of quantum communication. Candidates were prioritized by relevance, citation impact, and recency rather than read exhaustively. A check against 31 references already known to be relevant showed that the searches retrieved 24 of them (7

In [10]:
# Cell 10: generate ref.bib from the curated list and verify every entry (Crossref and arXiv)
def ascii_lower(s): return unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
def clean(s): return re.sub(r"\s+", " ", re.sub(r"[{}]", "", re.sub(r"\\[a-zA-Z]+\s*", "", str(s or "")))).strip()
def first_surname(a):
    a = clean(str(a).split(" and ")[0])
    return ascii_lower(a.split(",")[0] if "," in a else a.split(" ")[-1])

CR = "https://api.crossref.org/works"
def crossref_by_doi(doi):
    r = requests.get(f"{CR}/{doi}", params={"mailto": EMAIL}, headers=HEADERS, timeout=30)
    return r.json()["message"] if r.status_code == 200 else None
def crossref_by_title(title, author, year):
    params = {"query.bibliographic": f"{clean(title)} {author}", "rows": 3, "mailto": EMAIL}
    if year == year: params["filter"] = f"from-pub-date:{int(year)-1},until-pub-date:{int(year)+1}"
    r = requests.get(CR, params=params, headers=HEADERS, timeout=30)
    items = r.json()["message"]["items"] if r.status_code == 200 else []
    return max(items, key=lambda it: fuzz.token_set_ratio(clean(title).lower(), clean((it.get("title") or [""])[0]).lower())) if items else None
def arxiv_by_id(aid):
    r = requests.get("http://export.arxiv.org/api/query", params={"id_list": aid}, headers=HEADERS, timeout=30)
    ent = ET.fromstring(r.text).find("a:entry", NS) if r.status_code == 200 else None
    if ent is None or ent.find("a:title", NS) is None: return None
    return dict(title=re.sub(r"\s+", " ", ent.find("a:title", NS).text).strip(), year=int(ent.find("a:published", NS).text[:4]),
                authors=[a.find("a:name", NS).text for a in ent.findall("a:author", NS)])

def cr_authors(item):
    return [f"{a.get('family','')}, {a.get('given','')}".strip(", ") for a in item.get("author", []) if a.get("family") or a.get("name")]

records, bib_rows = [], []
for _, r in tqdm(curated.iterrows(), total=len(curated)):
    rec = dict(key=r.key, title=r.title, year=r.year, doi=r.doi, method="", match_title="", match_year="", title_sim=0, status="SKIPPED")
    authors, doi, volume, number, pages, container = r.authors, r.doi, r.volume, r.number, r.pages, r.container
    if VERIFY_ONLINE:
        try:
            am = re.search(r"arxiv[:\s]*([0-9]{4}\.[0-9]{4,5})", str(r.note), re.I)
            item = None
            if am:
                rec["method"] = "arxiv-id"; it = arxiv_by_id(am.group(1))
                if it:
                    rec.update(match_title=it["title"], match_year=it["year"]); item = ("arxiv", it)
            elif "ePrint" in str(r.note) or "eprint" in str(r.note).lower():
                rec["method"] = "iacr-eprint"; rec["status"] = "MANUAL"
            elif r.doi:
                rec["method"] = "crossref-doi"; it = crossref_by_doi(r.doi)
                if it: item = ("crossref", it)
            else:
                rec["method"] = "crossref-title"; it = crossref_by_title(r.title, first_surname(r.authors), r.year)
                if it: item = ("crossref", it)
            if item:
                kind, it = item
                mt = it["title"] if kind == "arxiv" else (it.get("title") or [""])[0]
                my = it["year"] if kind == "arxiv" else (it.get("issued", {}).get("date-parts") or [[None]])[0][0]
                fams = [ascii_lower(a.split(" ")[-1]) for a in it["authors"]] if kind == "arxiv" else [ascii_lower(a.get("family", "")) for a in it.get("author", [])]
                sim = fuzz.token_set_ratio(clean(r.title).lower(), clean(mt).lower())
                year_ok = (abs(int(r.year) - int(my)) <= 1) if (my and r.year == r.year) else None
                author_ok = (first_surname(r.authors) in fams) if fams else None
                rec.update(match_title=mt, match_year=my, title_sim=sim)
                if sim >= 90 and year_ok is not False and author_ok is not False: rec["status"] = "VERIFIED"
                elif sim >= 75: rec["status"] = "CHECK"
                else: rec["status"] = "NOT_FOUND"
                if rec["status"] == "VERIFIED" and kind == "crossref":      # enrich the entry from the verified record
                    names = cr_authors(it)
                    if names: authors = " and ".join(names[:MAX_AUTHORS]) + (" and others" if len(names) > MAX_AUTHORS else "")
                    doi = doi or it.get("DOI", "").lower()
                    volume = volume or it.get("volume", ""); number = number or it.get("issue", ""); pages = pages or it.get("page", "")
                    container = container or (it.get("container-title") or [""])[0]
                elif rec["status"] == "VERIFIED" and kind == "arxiv":
                    names = it["authors"]; authors = " and ".join(names[:MAX_AUTHORS]) + (" and others" if len(names) > MAX_AUTHORS else "")
            elif rec["status"] == "SKIPPED":
                rec["status"] = "NOT_FOUND"
        except Exception as ex:
            rec["status"] = f"ERROR:{type(ex).__name__}"
        time.sleep(0.1)
    records.append(rec)
    bib_rows.append(dict(r, authors=authors, doi=doi, volume=volume, number=number, pages=pages, container=container))

ver = pd.DataFrame(records); ver.to_csv(f"{OUT}/refs_verification.csv", index=False)
print(ver.status.value_counts().to_dict())
display(ver[~ver.status.isin(["VERIFIED", "SKIPPED"])][["key", "year", "status", "title_sim", "match_year", "title", "match_title"]])

def bibtex(r):
    typ = r["entry_type"]; cont_field = "journal" if typ == "article" else ("booktitle" if typ in ("inproceedings", "incollection") else None)
    f = [("author", r["authors"]), ("title", r["title"])]
    if r["container"] and cont_field: f.append((cont_field, r["container"]))
    for k in ("publisher", "volume", "number", "pages", "year", "doi", "note"):
        v = r[k] if k != "year" else (int(r["year"]) if r["year"] == r["year"] else "")
        if str(v).strip() not in ("", "nan"): f.append((k, v))
    return f"@{typ}{{{r['key']},\n" + ",\n".join(f"  {k} = {{{v}}}" for k, v in f) + "\n}\n"
with open(f"{OUT}/ref.bib", "w", encoding="utf-8") as fh:
    fh.write("\n".join(bibtex(r) for r in bib_rows))
print(f"ref.bib written with {len(bib_rows)} entries ->", f"{OUT}/ref.bib")

  0%|          | 0/126 [00:00<?, ?it/s]

{'VERIFIED': 120, 'NOT_FOUND': 3, 'CHECK': 2, 'MANUAL': 1}


,key,year,status,title_sim,match_year,title,match_title
17,Ref18,2023,CHECK,100.0,2022,Measurement-induced entanglement and teleporta...,Quantum entanglement and teleportation
33,Ref34,2020,CHECK,100.0,2020,Data re-uploading for a universal quantum clas...,Data re-uploading for a universal quantum clas...
67,Ref72,2017,NOT_FOUND,0.0,,Ground-to-satellite quantum teleportation,
69,Ref74,2005,NOT_FOUND,0.0,,Decoy state quantum key distribution,
70,Ref75,2012,NOT_FOUND,0.0,,Measurement-device-independent quantum key dis...,
106,Gupta2026,2026,MANUAL,0.0,,Physics-Aware Temporal Feature Engineering for...,


ref.bib written with 126 entries -> outputs/ref.bib


In [11]:
# Cell 11: save a snapshot and download everything
try:
    from google.colab import files
    with zipfile.ZipFile("methodology_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, fs in os.walk(OUT):
            for f in fs: z.write(os.path.join(root, f))
    files.download("methodology_outputs.zip")
except ImportError:
    print("Not running in Colab: outputs are in", OUT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>